## 5. Patch Extraction Adımı 

Hücre 1: Yama Kesme (Patch Extraction) ve Dolgu (Padding) Fonksiyonu

Bu hücrede, CNN ve ViT gibi derin öğrenme modellerinin "uzamsal bağlamı" (spatial context) öğrenebilmesi için gereken veri küplerini üreten temel fonksiyon tanımlanır.

Geometrik Dolgu (Reflect Padding): Haritanın en kenarındaki piksellerin de veri setine dahil edilebilmesi için uzamsal eksenlere yansıma yöntemiyle 7 piksellik çerçeve eklenir. Spektral bütünlüğü bozmamak için bant eksenine dokunulmaz.

Merkezlenmiş Yama Çıkarma: Etiketsiz arka plan pikselleri (0) elenir. Sadece sınıfı bilinen pikseller tam merkeze alınarak 15x15x30 boyutunda 3 boyutlu özellik (X) ve etiket (y) listeleri oluşturulur.

In [3]:
import numpy as np

def extract_patches(img, gt, window_size=15):
    margin = window_size // 2
    img_padded = np.pad(img, ((margin, margin), (margin, margin), (0, 0)), mode='reflect')
    
    rows, cols = np.where(gt > 0)
    X = np.stack([
        img_padded[r:r + window_size, c:c + window_size, :]
        for r, c in zip(rows, cols)
    ])
    y = gt[rows, cols]
    return X, y


Hücre 2: Otomatik İşleme, Tabakalı Bölme (Stratified Split) ve Diske Kayıt

Bu hücrede, ön işlemlerden geçmiş (Farklı Sigma + PCA30) 9 dosyanın tamamı otomatik bir döngüye sokularak nihai model girdileri hazırlanır.

Veri Eşleştirme ve Yama Üretimi: PCA dosyaları ile Ground Truth (GT) haritaları eşleştirilip birinci hücredeki fonksiyon çağrılır.

Sınıf Dengeli Ayrım (Stratify=y): Azınlık sınıflarının (Örn: Indian Pines'taki Yulaf sınıfı) eğitimde kaybolmasını önlemek için veriler %10 Eğitim ve %90 Test oranında, sınıf nüfuslarına sadık kalınarak bölünür.

Bellek Optimizasyonu (RAM Yönetimi): Üretilen yüksek boyutlu matrisler (X_train, X_test, vb.) doğrudan diske .npy olarak kaydedilir. Cihaz belleğinin (RAM) şişmesini önlemek için her adımda değişkenler silinir ve çöp toplayıcı (gc.collect()) çalıştırılır.

In [4]:
import os
import gc
from sklearn.model_selection import train_test_split

PROCESSED_DIR = './datasets/processed'

gt_map = {
    'indian_pines': np.load(os.path.join(PROCESSED_DIR, 'indian_pines_gt.npy')),
    'pavia_university': np.load(os.path.join(PROCESSED_DIR, 'pavia_university_gt.npy')),
    'salinas': np.load(os.path.join(PROCESSED_DIR, 'salinas_gt.npy')),
}

import glob
pca_files = sorted(glob.glob(os.path.join(PROCESSED_DIR, '*_gauss*_pca30.npy')))

for fpath in pca_files:
    fname = os.path.basename(fpath).replace('.npy', '')
    prefix = fname.split('_gauss')[0]

    img = np.load(fpath)
    gt  = gt_map[prefix]

    X, y = extract_patches(img, gt)
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.9, stratify=y, random_state=42
    )

    np.save(os.path.join(PROCESSED_DIR, f'{fname}_X_train.npy'), X_train)
    np.save(os.path.join(PROCESSED_DIR, f'{fname}_X_test.npy'),  X_test)
    np.save(os.path.join(PROCESSED_DIR, f'{fname}_y_train.npy'), y_train)
    np.save(os.path.join(PROCESSED_DIR, f'{fname}_y_test.npy'),  y_test)

    print(f"✓ {fname} → X_train:{X_train.shape} X_test:{X_test.shape}")

    del X, y, X_train, X_test, y_train, y_test
    gc.collect()


✓ indian_pines_gauss0.5_pca30 → X_train:(1024, 15, 15, 30) X_test:(9225, 15, 15, 30)
✓ pavia_university_gauss0.5_pca30 → X_train:(4277, 15, 15, 30) X_test:(38499, 15, 15, 30)
✓ salinas_gauss0.5_pca30 → X_train:(5412, 15, 15, 30) X_test:(48717, 15, 15, 30)
